# Day 2: Retrieval-Augmented Generation (RAG) & LangChain

**Workshop: Introduction to Generative AI & LLMs**

---

## Learning Objectives

By the end of this session, you will be able to:

1. Understand how text embeddings work and why they matter
2. Encode text and compute semantic similarity with sentence-transformers
3. Store and query embeddings with ChromaDB
4. Build a full RAG pipeline: load, chunk, embed, retrieve, generate
5. Use LangChain for chains, prompts, and output parsers
6. Build a Q&A system over a document collection

---

## 1. Embeddings: Concepts

### What Are Embeddings?

Embeddings are dense vector representations of text in a continuous vector space. Semantically similar texts have vectors that are close together.

```
"The cat sat on the mat"  -->  [0.12, -0.34, 0.78, ..., 0.56]  (768 dimensions)
"A kitten rested on a rug" -->  [0.11, -0.31, 0.75, ..., 0.54]  (similar vector!)
"Stock prices fell today"  -->  [-0.45, 0.67, -0.12, ..., 0.89]  (different vector)
```

### Why Embeddings Matter for RAG

LLMs have a finite context window and a knowledge cutoff. Embeddings let us:
1. **Search** over large document collections semantically (not just keyword matching)
2. **Retrieve** the most relevant passages for a query
3. **Augment** the LLM's prompt with retrieved context
4. **Generate** grounded answers based on actual documents

### The RAG Pipeline

```
                    INDEXING (offline)                      QUERYING (online)

Documents --> [Chunk] --> [Embed] --> Vector DB     User Query --> [Embed] --> Vector DB
                                        |                                       |
                                        +---- stored vectors ----+              |
                                                                  |              |
                                                                  +--> [Retrieve top-k]
                                                                           |
                                                              Retrieved chunks + Query
                                                                           |
                                                                      [LLM Generate]
                                                                           |
                                                                        Answer
```

### Popular Embedding Models

| Model | Dimensions | Provider | Notes |
|-------|-----------|----------|-------|
| all-MiniLM-L6-v2 | 384 | Sentence-Transformers | Fast, good quality |
| all-mpnet-base-v2 | 768 | Sentence-Transformers | Higher quality |
| text-embedding-3-small | 1536 | OpenAI | API-based, excellent quality |
| text-embedding-3-large | 3072 | OpenAI | Best quality, higher cost |
| BGE-large-en | 1024 | BAAI | Open-source, competitive |

---

## 2. Setup

In [ ]:
# Install required packages
!pip install sentence-transformers chromadb langchain langchain-openai langchain-community openai tiktoken --quiet

In [ ]:
import os
import json
import numpy as np
from textwrap import dedent

from dotenv import load_dotenv
load_dotenv()

# Verify OpenAI API key
api_key = os.environ.get("OPENAI_API_KEY")
print(f"OpenAI API key configured: {'Yes' if api_key else 'No -- please set OPENAI_API_KEY'}")

---

## 3. Sentence-Transformers: Encoding & Similarity

In [ ]:
from sentence_transformers import SentenceTransformer

# Load a pre-trained embedding model
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

# Encode some sentences
sentences = [
    "Machine learning is a subset of artificial intelligence.",
    "Deep learning uses neural networks with many layers.",
    "The weather in Paris is pleasant in spring.",
    "Natural language processing deals with text and speech.",
    "It is sunny and warm in the French capital during April.",
]

embeddings = embed_model.encode(sentences)

print(f"Number of sentences: {len(sentences)}")
print(f"Embedding shape: {embeddings.shape}")
print(f"Embedding dtype: {embeddings.dtype}")
print(f"\nFirst embedding (first 10 dims): {embeddings[0][:10]}")

In [ ]:
from sentence_transformers import util

# Compute cosine similarity between all pairs
cosine_scores = util.cos_sim(embeddings, embeddings)

print("Cosine Similarity Matrix:")
print("=" * 80)

# Print as a formatted table
for i in range(len(sentences)):
    print(f"\nSentence {i}: \"{sentences[i][:50]}...\"")
    for j in range(len(sentences)):
        if j > i:
            score = cosine_scores[i][j].item()
            marker = " <-- HIGH" if score > 0.5 else ""
            print(f"  vs Sentence {j}: {score:.4f}{marker}")

In [ ]:
# Semantic search: find the most similar sentence to a query
query = "How do computers understand human language?"
query_embedding = embed_model.encode(query)

# Compute similarity between query and all sentences
scores = util.cos_sim(query_embedding, embeddings)[0]

print(f"Query: \"{query}\"")
print("\nRanked results:")
print("-" * 60)

# Sort by score descending
ranked = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)
for rank, (idx, score) in enumerate(ranked, 1):
    print(f"  {rank}. [{score:.4f}] {sentences[idx]}")

---

## 4. Vector Database with ChromaDB

ChromaDB is a lightweight, open-source vector database that makes it easy to store, search, and manage embeddings.

In [ ]:
import chromadb

# Create an in-memory ChromaDB client
chroma_client = chromadb.Client()

# Create a collection (like a table) with a name
collection = chroma_client.create_collection(
    name="research_papers",
    metadata={"hnsw:space": "cosine"},  # Use cosine similarity
)

print(f"Collection created: {collection.name}")
print(f"Current count: {collection.count()}")

In [ ]:
# Sample research abstracts to index
research_abstracts = [
    {
        "id": "paper_001",
        "text": "We introduce a novel attention mechanism that reduces the quadratic "
                "complexity of standard self-attention to linear complexity. Our method, "
                "called FlashAttention, uses tiling and recomputation to minimize memory "
                "reads/writes between GPU high-bandwidth memory and on-chip SRAM. "
                "Experiments show 2-4x speedup over standard attention with no loss in quality.",
        "metadata": {"topic": "efficient transformers", "year": 2022, "venue": "NeurIPS"},
    },
    {
        "id": "paper_002",
        "text": "Retrieval-Augmented Generation (RAG) combines a pre-trained language model "
                "with a non-parametric retrieval module. Given a query, relevant documents are "
                "retrieved from a knowledge base and concatenated with the input. This approach "
                "reduces hallucination and allows the model to access up-to-date information "
                "without retraining.",
        "metadata": {"topic": "retrieval augmented generation", "year": 2020, "venue": "NeurIPS"},
    },
    {
        "id": "paper_003",
        "text": "We present LoRA (Low-Rank Adaptation), a parameter-efficient fine-tuning "
                "method that freezes the pre-trained model weights and injects trainable "
                "low-rank decomposition matrices into each Transformer layer. LoRA reduces "
                "the number of trainable parameters by 10,000x and GPU memory by 3x compared "
                "to full fine-tuning, with comparable performance.",
        "metadata": {"topic": "efficient fine-tuning", "year": 2021, "venue": "ICLR"},
    },
    {
        "id": "paper_004",
        "text": "We study the problem of training language models with human feedback. "
                "Our approach, RLHF, first trains a reward model from human preferences, "
                "then uses PPO to fine-tune the language model to maximize the learned reward. "
                "This significantly improves the helpfulness and safety of model outputs "
                "compared to supervised fine-tuning alone.",
        "metadata": {"topic": "alignment", "year": 2022, "venue": "NeurIPS"},
    },
    {
        "id": "paper_005",
        "text": "Vector databases have become essential infrastructure for AI applications. "
                "We benchmark five popular vector databases (Pinecone, Weaviate, Milvus, Qdrant, "
                "ChromaDB) on recall, latency, and throughput across different dataset sizes. "
                "Results show that approximate nearest-neighbor search with HNSW indexing "
                "provides the best balance of speed and accuracy for most use cases.",
        "metadata": {"topic": "vector databases", "year": 2023, "venue": "VLDB"},
    },
    {
        "id": "paper_006",
        "text": "We propose a new chunking strategy for RAG systems that respects document "
                "structure. Instead of fixed-size character splits, our method identifies "
                "semantic boundaries using sentence embeddings and creates variable-length "
                "chunks that preserve coherent units of meaning. This improves retrieval "
                "precision by 15% on the BEIR benchmark.",
        "metadata": {"topic": "retrieval augmented generation", "year": 2024, "venue": "ACL"},
    },
    {
        "id": "paper_007",
        "text": "Chain-of-Thought prompting enables large language models to solve complex "
                "reasoning tasks by generating intermediate reasoning steps. We show that "
                "this technique, combined with self-consistency (sampling multiple reasoning "
                "paths and taking the majority vote), achieves state-of-the-art on GSM8K, "
                "ARC, and StrategyQA benchmarks.",
        "metadata": {"topic": "prompting", "year": 2023, "venue": "NeurIPS"},
    },
    {
        "id": "paper_008",
        "text": "We introduce Mixtral 8x7B, a sparse mixture-of-experts (MoE) language model. "
                "Each token is routed to 2 of 8 expert sub-networks, allowing the model to "
                "have 47B total parameters while only using 13B per forward pass. Mixtral "
                "matches or exceeds GPT-3.5 and LLaMA 2 70B on most benchmarks while being "
                "significantly faster at inference.",
        "metadata": {"topic": "model architecture", "year": 2024, "venue": "arXiv"},
    },
]

# Encode all abstracts
abstract_texts = [p["text"] for p in research_abstracts]
abstract_embeddings = embed_model.encode(abstract_texts).tolist()

# Add to ChromaDB
collection.add(
    ids=[p["id"] for p in research_abstracts],
    documents=abstract_texts,
    embeddings=abstract_embeddings,
    metadatas=[p["metadata"] for p in research_abstracts],
)

print(f"Added {collection.count()} documents to the collection.")

In [ ]:
# Query the collection
def search_papers(query: str, n_results: int = 3, where: dict = None):
    """Search the research papers collection."""
    query_embedding = embed_model.encode(query).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results,
        where=where,
    )

    print(f"Query: \"{query}\"")
    if where:
        print(f"Filter: {where}")
    print("-" * 60)

    for i in range(len(results["ids"][0])):
        doc_id = results["ids"][0][i]
        distance = results["distances"][0][i]
        doc = results["documents"][0][i]
        meta = results["metadatas"][0][i]
        print(f"\n  [{i+1}] {doc_id} (distance: {distance:.4f})")
        print(f"      Topic: {meta['topic']} | Year: {meta['year']} | Venue: {meta['venue']}")
        print(f"      {doc[:120]}...")

    return results


# Search for RAG-related papers
results = search_papers("How to reduce hallucination in language models using external knowledge?")

In [ ]:
# Search with metadata filtering
print("\n" + "=" * 60)
results = search_papers(
    "efficient model training",
    n_results=3,
    where={"year": {"$gte": 2023}},  # Only papers from 2023 or later
)

In [ ]:
# Search for a different topic
print("\n" + "=" * 60)
results = search_papers("How to make language models safer and aligned with human values?")

---

## 5. Building a RAG Pipeline from Scratch

Now let's build a complete RAG pipeline step by step.

In [ ]:
from openai import OpenAI

openai_client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))


# Step 1: Document Loading (using inline sample data)
# In production, you would load from files, URLs, databases, etc.

documents = [
    {
        "source": "ai_safety_report.txt",
        "content": dedent("""\
            AI Safety: Current Challenges and Future Directions

            The rapid deployment of large language models has raised significant safety concerns.
            Key challenges include hallucination, where models generate plausible but factually
            incorrect information. Studies show that even the most capable models hallucinate
            in 3-15% of responses depending on the domain.

            Alignment techniques such as RLHF and Constitutional AI have shown promise in
            reducing harmful outputs. However, these methods are not foolproof. Red-teaming
            exercises have revealed that determined adversaries can still elicit problematic
            content through creative prompt injection and jailbreaking techniques.

            Emerging approaches include mechanistic interpretability, which aims to understand
            the internal representations of neural networks, and formal verification methods
            that provide mathematical guarantees about model behavior in constrained settings.

            The AI safety community recommends a defense-in-depth strategy: combining multiple
            safety layers including input filtering, output monitoring, human-in-the-loop
            review for high-stakes decisions, and robust evaluation frameworks."""),
    },
    {
        "source": "rag_best_practices.txt",
        "content": dedent("""\
            Best Practices for RAG Systems

            Retrieval-Augmented Generation has become a standard pattern for building
            knowledge-grounded AI applications. Here are key best practices:

            Chunking Strategy: Choose chunk sizes that balance context and precision.
            Typical sizes range from 256 to 1024 tokens. Overlap between chunks (usually
            50-100 tokens) helps preserve context at boundaries. Consider semantic chunking
            that respects paragraph and section boundaries.

            Embedding Model Selection: Use domain-specific embedding models when available.
            For general use, models like text-embedding-3-small offer an excellent balance
            of quality and cost. Always evaluate on your specific data.

            Retrieval Optimization: Start with top-k retrieval (k=3-5) and adjust based
            on results. Consider hybrid search combining dense (vector) and sparse (BM25)
            retrieval for better coverage. Re-ranking retrieved results with a cross-encoder
            can significantly improve precision.

            Prompt Design: Include clear instructions for the LLM to use only the provided
            context. Ask the model to cite sources and indicate when it cannot find relevant
            information. This reduces hallucination and improves trustworthiness.

            Evaluation: Use metrics like faithfulness (does the answer match the retrieved
            context?), relevance (are retrieved documents relevant to the query?), and
            answer correctness. Tools like RAGAS provide automated evaluation frameworks."""),
    },
    {
        "source": "llm_deployment.txt",
        "content": dedent("""\
            Deploying LLMs in Production

            Moving from prototype to production with LLMs requires careful consideration
            of several factors.

            Latency: Users expect sub-second response times for interactive applications.
            Techniques to reduce latency include model quantization (4-bit or 8-bit),
            speculative decoding, KV-cache optimization, and streaming responses.
            For batch processing, throughput optimization through continuous batching
            is more important than individual request latency.

            Cost Management: API costs can escalate quickly at scale. Strategies include
            using smaller models for simple tasks (routing), caching frequent queries,
            prompt optimization to reduce token count, and self-hosting open models
            for high-volume use cases.

            Monitoring: Track key metrics including response latency, token usage,
            error rates, and output quality. Implement logging for all LLM interactions
            to enable debugging and improvement. Use LLM-as-judge evaluation for
            automated quality monitoring.

            Scaling: Use autoscaling based on request queue depth. Consider serverless
            GPU options for variable workloads. For self-hosted models, tools like
            vLLM and TGI provide optimized serving with continuous batching."""),
    },
]

print(f"Loaded {len(documents)} documents")
for doc in documents:
    print(f"  - {doc['source']}: {len(doc['content'])} characters")

In [ ]:
# Step 2: Chunking

def chunk_text(text: str, chunk_size: int = 300, overlap: int = 50) -> list[str]:
    """Split text into overlapping chunks by character count."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        # Try to break at a sentence boundary
        if end < len(text):
            # Look for the last period within the chunk
            last_period = text.rfind(".", start, end)
            if last_period > start + chunk_size // 2:
                end = last_period + 1
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        start = end - overlap
    return chunks


# Chunk all documents
all_chunks = []
chunk_metadata = []

for doc in documents:
    chunks = chunk_text(doc["content"], chunk_size=400, overlap=50)
    for i, chunk in enumerate(chunks):
        all_chunks.append(chunk)
        chunk_metadata.append({
            "source": doc["source"],
            "chunk_index": i,
        })

print(f"Total chunks: {len(all_chunks)}")
print("\nSample chunks:")
for i, (chunk, meta) in enumerate(zip(all_chunks[:3], chunk_metadata[:3])):
    print(f"\n  Chunk {i} ({meta['source']}, idx {meta['chunk_index']}):")
    print(f"  {chunk[:150]}...")

In [ ]:
# Step 3: Embed and store in ChromaDB

# Create a new collection for our RAG system
rag_collection = chroma_client.create_collection(
    name="rag_knowledge_base",
    metadata={"hnsw:space": "cosine"},
)

# Embed all chunks
chunk_embeddings = embed_model.encode(all_chunks).tolist()

# Add to ChromaDB
rag_collection.add(
    ids=[f"chunk_{i}" for i in range(len(all_chunks))],
    documents=all_chunks,
    embeddings=chunk_embeddings,
    metadatas=chunk_metadata,
)

print(f"Indexed {rag_collection.count()} chunks in ChromaDB.")

In [ ]:
# Step 4: Retrieval

def retrieve(query: str, n_results: int = 3) -> list[dict]:
    """Retrieve the most relevant chunks for a query."""
    query_embedding = embed_model.encode(query).tolist()

    results = rag_collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results,
    )

    retrieved = []
    for i in range(len(results["ids"][0])):
        retrieved.append({
            "id": results["ids"][0][i],
            "text": results["documents"][0][i],
            "metadata": results["metadatas"][0][i],
            "distance": results["distances"][0][i],
        })

    return retrieved


# Test retrieval
query = "What are the best practices for chunking documents in RAG?"
retrieved_chunks = retrieve(query, n_results=3)

print(f"Query: {query}\n")
for i, chunk in enumerate(retrieved_chunks):
    print(f"Result {i+1} (distance: {chunk['distance']:.4f}, source: {chunk['metadata']['source']}):")
    print(f"  {chunk['text'][:200]}...")
    print()

In [ ]:
# Step 5: Generation with retrieved context

RAG_SYSTEM_PROMPT = """You are a knowledgeable AI assistant. Answer the user's question 
based ONLY on the provided context. If the context does not contain enough information 
to answer, say so clearly. Always cite the source document when possible.

Format your answer clearly with paragraphs. Be concise but thorough."""

RAG_USER_TEMPLATE = """Context:
{context}

Question: {question}

Answer:"""


def rag_answer(question: str, n_chunks: int = 3) -> str:
    """Full RAG pipeline: retrieve context and generate an answer."""
    # Retrieve
    chunks = retrieve(question, n_results=n_chunks)

    # Build context string
    context_parts = []
    for i, chunk in enumerate(chunks):
        context_parts.append(
            f"[Source: {chunk['metadata']['source']}]\n{chunk['text']}"
        )
    context = "\n\n---\n\n".join(context_parts)

    # Generate
    user_message = RAG_USER_TEMPLATE.format(context=context, question=question)

    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": RAG_SYSTEM_PROMPT},
            {"role": "user", "content": user_message},
        ],
        temperature=0.3,
        max_tokens=500,
    )

    return response.choices[0].message.content


# Test the full RAG pipeline
question = "How can I reduce latency when deploying an LLM in production?"
answer = rag_answer(question)

print(f"Question: {question}")
print(f"\nAnswer:\n{answer}")

In [ ]:
# Try more questions
questions = [
    "What are the main approaches to AI safety?",
    "How should I choose the chunk size for my RAG system?",
    "What is the recommended strategy for managing LLM API costs?",
]

for q in questions:
    print("=" * 70)
    answer = rag_answer(q)
    print(f"Q: {q}")
    print(f"\nA: {answer}\n")

---

## 6. LangChain Basics

LangChain provides higher-level abstractions for building LLM-powered applications. Let's explore its core components.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from langchain_core.pydantic_v1 import BaseModel, Field

# Initialize the LLM
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    api_key=os.environ.get("OPENAI_API_KEY"),
)

# Test basic invocation
response = llm.invoke("What is RAG in one sentence?")
print(f"Type: {type(response)}")
print(f"Content: {response.content}")

In [ ]:
# LangChain Prompt Templates

# Simple prompt template
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert in {domain}. Be concise."),
    ("human", "Explain {concept} in simple terms."),
])

# Create a chain: prompt -> LLM -> output parser
chain = prompt | llm | StrOutputParser()

# Invoke the chain
result = chain.invoke({"domain": "machine learning", "concept": "gradient descent"})
print("Chain result:")
print(result)

In [ ]:
# Structured Output with Pydantic models

class PaperSummary(BaseModel):
    """Summary of an academic paper."""
    title: str = Field(description="Title of the paper")
    key_contribution: str = Field(description="The main contribution in one sentence")
    methodology: str = Field(description="Brief description of the methodology")
    limitations: list[str] = Field(description="List of limitations or caveats")
    relevance_score: int = Field(description="Relevance to AI practitioners (1-10)")


# Create a JSON output parser
json_parser = JsonOutputParser(pydantic_object=PaperSummary)

# Build the chain
extract_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an academic paper analysis assistant."),
    ("human", "Analyze this paper abstract and extract structured information.\n\n"
              "Abstract: {abstract}\n\n{format_instructions}"),
])

extract_chain = extract_prompt | llm | json_parser

# Run extraction
abstract = """
We present LoRA (Low-Rank Adaptation), a parameter-efficient fine-tuning method that 
freezes the pre-trained model weights and injects trainable low-rank decomposition 
matrices into each Transformer layer. LoRA reduces the number of trainable parameters 
by 10,000x and GPU memory by 3x compared to full fine-tuning, with comparable 
performance on RoBERTa, DeBERTa, GPT-2, and GPT-3.
"""

result = extract_chain.invoke({
    "abstract": abstract,
    "format_instructions": json_parser.get_format_instructions(),
})

print("Extracted paper summary:")
print(json.dumps(result, indent=2))

In [ ]:
# Sequential Chain: Multi-step processing
from langchain_core.runnables import RunnablePassthrough

# Step 1: Generate a research question
question_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a research advisor."),
    ("human", "Generate one specific research question about {topic}. "
              "Return ONLY the question, nothing else."),
])

# Step 2: Outline the approach
outline_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a research methodology expert."),
    ("human", "For this research question: {question}\n\n"
              "Provide a brief methodology outline (3-4 steps)."),
])

# Chain them together
question_chain = question_prompt | llm | StrOutputParser()
outline_chain = outline_prompt | llm | StrOutputParser()

# Combined chain
full_chain = (
    {"question": question_chain}
    | RunnablePassthrough.assign(outline=outline_chain)
)

result = full_chain.invoke({"topic": "improving RAG systems for scientific literature"})
print("Generated research question:")
print(result["question"])
print("\nMethodology outline:")
print(result["outline"])

---

## 7. Building a Q&A System with LangChain

Let's build a proper Q&A system using LangChain's RAG components.

In [ ]:
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import SentenceTransformerEmbeddings
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Create LangChain-compatible embedding function
lc_embeddings = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")

# Convert our documents to LangChain Document objects
lc_documents = []
for doc in documents:
    # Split into chunks
    chunks = chunk_text(doc["content"], chunk_size=400, overlap=50)
    for i, chunk in enumerate(chunks):
        lc_documents.append(Document(
            page_content=chunk,
            metadata={"source": doc["source"], "chunk": i},
        ))

# Create a Chroma vector store from documents
vectorstore = Chroma.from_documents(
    documents=lc_documents,
    embedding=lc_embeddings,
    collection_name="langchain_rag",
)

# Create a retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print(f"Vector store created with {len(lc_documents)} chunks.")
print(f"Retriever ready.")

In [ ]:
# Build the RAG chain

# Prompt template for RAG
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI assistant. Answer based ONLY on the provided context. "
               "If you cannot answer from the context, say 'I don't have enough information "
               "to answer this question.' Cite sources when possible."),
    ("human", "Context:\n{context}\n\nQuestion: {question}\n\nAnswer:"),
])


def format_docs(docs):
    """Format retrieved documents into a context string."""
    formatted = []
    for doc in docs:
        source = doc.metadata.get("source", "unknown")
        formatted.append(f"[Source: {source}]\n{doc.page_content}")
    return "\n\n---\n\n".join(formatted)


# Build the chain using LCEL (LangChain Expression Language)
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

print("RAG chain built successfully.")

In [ ]:
# Test the Q&A system
questions = [
    "What are the main techniques for reducing hallucination in LLMs?",
    "How should I approach chunking for a RAG system?",
    "What tools can I use for serving open-source LLMs?",
    "What is the recommended chunk overlap size?",
]

for question in questions:
    print("=" * 70)
    print(f"Q: {question}")
    answer = rag_chain.invoke(question)
    print(f"\nA: {answer}\n")

---

## 8. Lab: RAG Chatbot over Research Abstracts

In this lab, we build a conversational RAG chatbot that can answer questions about a collection of research paper abstracts. The chatbot maintains conversation history for follow-up questions.

In [ ]:
# Extended collection of research abstracts
lab_abstracts = [
    {
        "title": "Attention Is All You Need",
        "year": 2017,
        "abstract": "We propose a new simple network architecture, the Transformer, based "
                     "solely on attention mechanisms, dispensing with recurrence and convolutions "
                     "entirely. The Transformer allows for significantly more parallelization "
                     "and can reach a new state of the art in translation quality after being "
                     "trained for as little as twelve hours on eight GPUs.",
    },
    {
        "title": "BERT: Pre-training of Deep Bidirectional Transformers",
        "year": 2019,
        "abstract": "We introduce BERT, a method for pre-training language representations. "
                     "BERT is designed to pre-train deep bidirectional representations from "
                     "unlabeled text by jointly conditioning on both left and right context. "
                     "The pre-trained BERT model can be fine-tuned with just one additional "
                     "output layer for a wide range of tasks.",
    },
    {
        "title": "GPT-3: Language Models are Few-Shot Learners",
        "year": 2020,
        "abstract": "We demonstrate that scaling up language models greatly improves "
                     "task-agnostic, few-shot performance. GPT-3, a 175 billion parameter "
                     "autoregressive language model, achieves strong performance on many NLP "
                     "benchmarks without any gradient updates or fine-tuning, using only "
                     "textual interactions to specify tasks.",
    },
    {
        "title": "LoRA: Low-Rank Adaptation of Large Language Models",
        "year": 2021,
        "abstract": "We propose Low-Rank Adaptation (LoRA) that freezes pre-trained model "
                     "weights and injects trainable rank decomposition matrices into each "
                     "layer of the Transformer architecture. LoRA reduces trainable parameters "
                     "by 10,000x and GPU memory requirement by 3x. It performs on par with "
                     "full fine-tuning on RoBERTa, DeBERTa, GPT-2, and GPT-3.",
    },
    {
        "title": "Constitutional AI: Harmlessness from AI Feedback",
        "year": 2022,
        "abstract": "We propose Constitutional AI (CAI), a method for training harmless AI "
                     "assistants without human feedback labels for harms. The method uses a "
                     "set of principles (a 'constitution') to generate self-critiques and "
                     "revisions, then uses reinforcement learning from AI feedback (RLAIF) "
                     "to further improve the model.",
    },
    {
        "title": "Retrieval-Augmented Generation for Knowledge-Intensive Tasks",
        "year": 2020,
        "abstract": "We explore a general-purpose fine-tuning recipe for retrieval-augmented "
                     "generation (RAG). RAG models combine pre-trained parametric and "
                     "non-parametric memory for language generation. The parametric memory is "
                     "a seq2seq model and the non-parametric memory is a dense vector index "
                     "of Wikipedia, accessed with a pre-trained neural retriever.",
    },
]

print(f"Lab dataset: {len(lab_abstracts)} research abstracts")
for paper in lab_abstracts:
    print(f"  - {paper['title']} ({paper['year']})")

In [ ]:
# Create the lab vector store
lab_docs = []
for paper in lab_abstracts:
    content = f"Title: {paper['title']}\nYear: {paper['year']}\n\n{paper['abstract']}"
    lab_docs.append(Document(
        page_content=content,
        metadata={"title": paper["title"], "year": paper["year"]},
    ))

lab_vectorstore = Chroma.from_documents(
    documents=lab_docs,
    embedding=lc_embeddings,
    collection_name="lab_papers",
)

lab_retriever = lab_vectorstore.as_retriever(search_kwargs={"k": 3})
print(f"Lab vector store created with {len(lab_docs)} documents.")

In [ ]:
# Conversational RAG chatbot with history

class RAGChatbot:
    """A conversational RAG chatbot over research papers."""

    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []

        self.system_prompt = dedent("""\
            You are a research paper expert assistant. You answer questions about 
            academic papers based on the provided context (retrieved paper abstracts).

            Rules:
            - Only use information from the provided context
            - Always mention the paper title when referencing a specific paper
            - If you cannot answer from the context, say so
            - Be concise but informative
            - Consider the conversation history for follow-up questions""")

    def _build_messages(self, question: str, context: str) -> list:
        """Build the full message list including history."""
        messages = [{"role": "system", "content": self.system_prompt}]

        # Add conversation history (last 6 turns to stay within context)
        for turn in self.history[-6:]:
            messages.append({"role": "user", "content": turn["question"]})
            messages.append({"role": "assistant", "content": turn["answer"]})

        # Add current question with context
        user_msg = f"Retrieved context:\n{context}\n\nQuestion: {question}"
        messages.append({"role": "user", "content": user_msg})

        return messages

    def ask(self, question: str) -> str:
        """Ask a question and get a RAG-powered answer."""
        # Retrieve relevant documents
        docs = self.retriever.invoke(question)
        context = format_docs(docs)

        # Build messages with history
        messages = self._build_messages(question, context)

        # Generate answer
        response = self.llm.invoke(messages)
        answer = response.content

        # Save to history
        self.history.append({"question": question, "answer": answer})

        return answer

    def reset(self):
        """Clear conversation history."""
        self.history = []
        print("Conversation history cleared.")


# Initialize the chatbot
chatbot = RAGChatbot(retriever=lab_retriever, llm=llm)
print("RAG Chatbot initialized and ready.")

In [ ]:
# Simulate a conversation
conversation = [
    "What papers discuss efficient fine-tuning of language models?",
    "How does LoRA specifically reduce memory requirements?",  # Follow-up
    "Are there papers about making AI systems safer?",
    "How does the Constitutional AI approach compare to RLHF?",  # Follow-up
    "Which paper introduced the Transformer architecture?",
]

for question in conversation:
    print("=" * 70)
    print(f"User: {question}")
    answer = chatbot.ask(question)
    print(f"\nAssistant: {answer}\n")

---

## Summary

In this session, we covered:

1. **Embeddings**: Dense vector representations for semantic similarity
2. **Sentence-Transformers**: Encoding text and computing similarity scores
3. **ChromaDB**: Creating collections, adding documents with metadata, querying
4. **RAG Pipeline**: Document loading, chunking, embedding, retrieval, generation
5. **LangChain**: Chains, prompt templates, output parsers, LCEL
6. **Q&A System**: Full RAG chain with LangChain components
7. **Conversational RAG**: Chatbot with history over research abstracts

### Next Session

In Day 3, we will explore **fine-tuning** LLMs, evaluation metrics, deploying with Gradio, and responsible AI practices.